# Hidden Markov Models

**Companion lesson:** https://ml-viz.vercel.app/courses/graphical-models/03-hidden-markov-models

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## The weather HMM

Hidden states {Rainy, Sunny}; observations {Walk, Shop, Clean}. We implement the **forward** (likelihood), **Viterbi** (best path) and **backward** algorithms from scratch.

In [ ]:
pi = np.array([0.6, 0.4])                       # P(z_1)
A  = np.array([[0.7, 0.3], [0.4, 0.6]])          # transitions
B  = np.array([[0.1, 0.4, 0.5], [0.6, 0.3, 0.1]])# emissions: rows=state, cols=obs
states = ['Rainy', 'Sunny']; obs_names = ['Walk', 'Shop', 'Clean']
obs = [0, 2, 1, 0]                               # Walk, Clean, Shop, Walk
print('observation sequence:', [obs_names[o] for o in obs])

## Forward algorithm — P(observations)

$\alpha_t(j)=B_{j,x_t}\sum_i \alpha_{t-1}(i)A_{ij}$, in $O(K^2T)$ instead of $K^T$.

In [ ]:
def forward(obs, pi, A, B):
    T, K = len(obs), len(pi)
    alpha = np.zeros((T, K))
    alpha[0] = pi * B[:, obs[0]]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A) * B[:, obs[t]]
    return alpha, alpha[-1].sum()

alpha, likelihood = forward(obs, pi, A, B)
print('P(observations) =', round(likelihood, 6))

## Viterbi — the single most likely hidden path

Same recursion with $\max$ instead of $\sum$, plus back-pointers to reconstruct the path.

In [ ]:
def viterbi(obs, pi, A, B):
    T, K = len(obs), len(pi)
    delta = np.zeros((T, K)); psi = np.zeros((T, K), int)
    delta[0] = np.log(pi) + np.log(B[:, obs[0]])     # log space avoids underflow
    for t in range(1, T):
        for j in range(K):
            scores = delta[t-1] + np.log(A[:, j])
            psi[t, j] = scores.argmax()
            delta[t, j] = scores.max() + np.log(B[j, obs[t]])
    path = [int(delta[-1].argmax())]
    for t in range(T-1, 0, -1):
        path.insert(0, psi[t, path[0]])
    return path, delta[-1].max()

path, logp = viterbi(obs, pi, A, B)
print('most likely weather:', [states[s] for s in path])
print('log-probability of that path:', round(logp, 3))

## Backward algorithm and posterior state probabilities

Combine forward and backward to get $P(z_t \mid \text{all observations})$ — the smoothed belief about each day's weather.

In [ ]:
def backward(obs, A, B):
    T, K = len(obs), A.shape[0]
    beta = np.zeros((T, K)); beta[-1] = 1
    for t in range(T-2, -1, -1):
        beta[t] = (A * B[:, obs[t+1]] * beta[t+1]).sum(axis=1)
    return beta

beta = backward(obs, A, B)
posterior = alpha * beta
posterior /= posterior.sum(axis=1, keepdims=True)
print('P(state | all obs) per day (Rainy, Sunny):')
for t, o in enumerate(obs):
    print(f'  day {t} ({obs_names[o]:5s}): {np.round(posterior[t], 3)}')

## Key takeaways

- **Forward** sums over hidden paths to get the sequence likelihood in $O(K^2T)$.
- **Viterbi** swaps sum for max (in log space) to recover the single best path.
- **Forward × backward** gives smoothed posteriors over each hidden state.
- These same recursions, wrapped in EM, become Baum-Welch for learning the parameters.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The forward algorithm

Implement the forward recursion on a small 2-state HMM:

$$\alpha_1 = \pi \odot B_{:,o_1}, \qquad \alpha_{t} = (\alpha_{t-1} A) \odot B_{:, o_t}, \qquad P(\text{obs}) = \sum_s \alpha_T(s)$$

The checks compare it against **brute-force enumeration over every hidden path** — same number, exponentially less work.

In [ ]:
A = np.array([[0.7, 0.3], [0.4, 0.6]])      # transitions
B = np.array([[0.9, 0.1], [0.2, 0.8]])      # emissions: B[state, observation]
pi0 = np.array([0.5, 0.5])


def forward(obs):
    """P(observation sequence) by the forward algorithm."""
    # TODO(you): initialize alpha with pi0 * B[:, obs[0]]
    alpha = ...

    for o in obs[1:]:
        # TODO(you): the recursion (alpha @ A) * B[:, o]
        alpha = ...

    return alpha.sum()

In [ ]:
# Checks — run me
def brute_force(obs):
    total, n = 0.0, len(obs)
    for path in range(2 ** n):
        states = [(path >> i) & 1 for i in range(n)]
        p = pi0[states[0]] * B[states[0], obs[0]]
        for t in range(1, n):
            p *= A[states[t - 1], states[t]] * B[states[t], obs[t]]
        total += p
    return total

for obs in [[0], [0, 1], [0, 1, 1], [1, 0, 1, 0]]:
    assert abs(forward(obs) - brute_force(obs)) < 1e-12, f"forward must equal path enumeration for {obs}"
assert abs(forward([0]) - (0.5 * 0.9 + 0.5 * 0.2)) < 1e-12, "one observation, by hand"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def forward(obs):
    alpha = pi0 * B[:, obs[0]]
    for o in obs[1:]:
        alpha = (alpha @ A) * B[:, o]
    return alpha.sum()
```

</details>

### Exercise 2 — Viterbi

Swap the forward algorithm's **sum** for a **max** (and remember which predecessor won) and you get the single most likely hidden path. The checks compare your decode against brute force over all paths — they must agree exactly.

In [ ]:
def viterbi(obs):
    """Most likely hidden state path for the observation sequence."""
    delta = pi0 * B[:, obs[0]]
    back = []

    for o in obs[1:]:
        cand = delta[:, None] * A          # cand[i, j] = delta[i] * A[i, j]

        # TODO(you): record the best predecessor of each state (argmax over axis 0)
        back.append(...)

        # TODO(you): the new delta: best incoming score per state, times the emission
        delta = ...

    # Backtrack
    path = [int(np.argmax(delta))]
    for bp in reversed(back):
        path.append(int(bp[path[-1]]))
    return list(reversed(path))

In [ ]:
# Checks — run me
def brute_best(obs):
    n, best, best_states = len(obs), -1.0, None
    for path in range(2 ** n):
        states = [(path >> i) & 1 for i in range(n)]
        p = pi0[states[0]] * B[states[0], obs[0]]
        for t in range(1, n):
            p *= A[states[t - 1], states[t]] * B[states[t], obs[t]]
        if p > best:
            best, best_states = p, states
    return best_states

for obs in [[0, 0], [0, 1, 1], [1, 1, 0, 0]]:
    assert viterbi(obs) == brute_best(obs), f"Viterbi must match brute force for {obs}"
assert viterbi([0, 0, 0]) == [0, 0, 0], "all-state-0 observations -> the all-0 path"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def viterbi(obs):
    delta = pi0 * B[:, obs[0]]
    back = []
    for o in obs[1:]:
        cand = delta[:, None] * A
        back.append(np.argmax(cand, axis=0))
        delta = cand.max(axis=0) * B[:, o]
    path = [int(np.argmax(delta))]
    for bp in reversed(back):
        path.append(int(bp[path[-1]]))
    return list(reversed(path))
```

</details>